<a href="https://colab.research.google.com/github/AdyashaGiri/Salesforcasting-simple-ML-project-/blob/main/SalesForcasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -Uq upgini catboost

In [ ]:
from os.path import exists

In [ ]:
import pandas as pd
from os.path import exists

df_path = "train.csv.zip" if exists("train.csv.zip") else "https://github.com/upgini/upgini/raw/main/notebooks/train.csv.zip"
df = pd.read_csv(df_path)
df = df.sample(n=19_000, random_state=0)
df["store"] = df["store"].astype(str)
df["item"] = df["item"].astype(str)

df["date"] = pd.to_datetime(df["date"])

df.sort_values("date", inplace=True)
df.reset_index(inplace=True, drop=True)
df.head()



In [ ]:
train = df[df["date"]< "2017-01-01"]
test= df[df["date"]>= "2017-01-01"]

In [ ]:
train_features = train.drop(columns=["sales"])
train_target = train["sales"]
test_features = test.drop(columns=["sales"])
test_target = test["sales"]

In [ ]:
from upgini import FeaturesEnricher, SearchKey
from upgini.metadata import CVType

# 1. Fixed the argument name to 'cv' and corrected 'time_series'
enricher = FeaturesEnricher(
    search_keys={
        "date": SearchKey.DATE,
    },
    cv=CVType.time_series
)

# 2. Fixed the eval_set syntax to correctly pass a list of tuples
enricher.fit(
    train_features,
    train_target,
    eval_set=[(test_features, test_target)]
)

In [ ]:
from catboost import CatBoostRegressor
from catboost.utils import eval_metric

model = CatBoostRegressor(verbose=False, allow_writing_files=False, random_state=0)

enricher.calculate_metrics(
    train_features, train_target,
    eval_set = [(test_features, test_target)],
    estimator = model,
    scoring = "mean_absolute_percentage_error"
)

In [ ]:
# 1. Take exactly what's left in your daily limit (906 rows)
train_features_subset = train_features.iloc[:900]
test_features_subset = test_features.iloc[:900]

# 2. Transform them and assign them to the variables
enriched_train_features = enricher.transform(train_features_subset, keep_input=True)
enriched_test_features = enricher.transform(test_features_subset, keep_input=True)

# 3. VERIFY THEY ARE NOT NONE
print("Train features type:", type(enriched_train_features))
print("Test features type:", type(enriched_test_features))

In [ ]:
model.fit(train_features, train_target)
preds = model.predict(test_features)
eval_metric(test_target.values, preds, "SMAPE")

In [ ]:
# Update targets to match the 906 rows
train_target_subset = train_target.iloc[:900]
test_target_subset = test_target.iloc[:900]

# Fit and predict
model.fit(enriched_train_features, train_target_subset)
enriched_preds = model.predict(enriched_test_features)

# Evaluate
eval_metric(test_target_subset.values, enriched_preds, "SMAPE")